In [ ]:
import numpy as np
from numpy.linalg import eigh
from scipy.linalg import eigh, expm

from qiskit import QuantumCircuit,  QuantumRegister, ClassicalRegister, AncillaRegister, transpile, qasm2
from qiskit.circuit import Parameter
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit.circuit.library import PauliEvolutionGate, hamiltonian_variational_ansatz
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit.synthesis import SuzukiTrotter
from qiskit.primitives import BackendEstimatorV2 as Estimator

import os

from tenpy.models.model import CouplingMPOModel
from tenpy.networks.site import SpinHalfFermionSite
from tenpy.models.lattice import Chain
from tenpy.networks.mps import MPS
from tenpy.algorithms import dmrg

import csv
import pandas as pd

import matplotlib.pyplot as plt
import matplotlib.ticker as tck

In [11]:
class AndersonImpurityModel(CouplingMPOModel):

    def init_sites(self, model_params):
        conserve = model_params.get('conserve', 'N')
        return SpinHalfFermionSite(cons_N=conserve)

    def init_lattice(self, model_params):
        L = model_params['L']
        site = self.init_sites(model_params)
        return Chain(L, site, bc='open')

    def init_terms(self, model_params):

        t = model_params['t']
        U = model_params['U']
        eps_d = model_params['eps_d']

        L = self.lat.N_sites
        imp = L // 2

        # nearest-neighbor hopping
        for i in range(L - 1):

            self.add_coupling(
                -t, i, 'Cdu', i + 1, 'Cu', dx = 1, 
                plus_hc=True
            )

            self.add_coupling(
                -t, i, 'Cdd', i + 1, 'Cd', dx = 1, 
                plus_hc=True
            )

        # impurity onsite energy
        self.add_onsite(eps_d, imp, 'Ntot')

        # impurity Hubbard interaction
        self.add_onsite(U, imp, 'NuNd')

In [12]:
model_params = {
    'L': 40,
    't': 1.0,
    'U': 8.0,
    'eps_d': -4.0,      # particle-hole symmetric point
    'conserve': 'N'
}

model = AndersonImpurityModel(model_params)

IndexError: list index out of range

In [ ]:
L = model.lat.N_sites

product_state = []

for i in range(L):
    if i % 2 == 0:
        product_state.append('up')
    else:
        product_state.append('down')

psi = MPS.from_product_state(
    model.lat.mps_sites(),
    product_state,
    bc='finite'
)

In [ ]:
dmrg_params = {
    'mixer': True,
    'trunc_params': {
        'chi_max': 200,
        'svd_min': 1e-10
    },
    'max_E_err': 1e-10,
    'max_sweeps': 20,
    'verbose': 1
}

info = dmrg.run(
    psi,
    model,
    dmrg_params
)

print("Ground-state energy:", info['E'])